In [ ]:
from datasets import load_dataset, Features, Value
import argparse
import json
import glob
import os
import numpy as np
import re
from pathlib import Path
import math

In [ ]:
result_dir = 'results'

model = 'Perception-Judge-Qwen3-4B'

temperature = 1.0
top_p = 0.95

seed = [0, 1, 2]

## Pair

In [ ]:
dss = []

try:
    ds1 = load_dataset('json', data_files=f'{result_dir}/mllm-judge-{temperature}-{top_p}/{model}_{seed[0]}_pair.jsonl', split='train')
    dss.append(ds1)
    print('Loaded ds1')
except:
    print('Can not load ds1')
try:
    ds2 = load_dataset('json', data_files=f'{result_dir}/mllm-judge-{temperature}-{top_p}/{model}_{seed[1]}_pair.jsonl', split='train')
    dss.append(ds2)
    print('Loaded ds2')
except:
    print('Can not load ds2')

try:
    ds3 = load_dataset('json', data_files=f'{result_dir}/mllm-judge-{temperature}-{top_p}/{model}_{seed[2]}_pair.jsonl', split='train')
    dss.append(ds3)
    print('Loaded ds3')
except:
    print('Can not load ds3')


In [23]:
import json
import numpy as np

def calculate_accuracy_metrics(ds, include_ties=False):
    """
    Calculate accuracy metrics between predictions and ground truth
    
    Args:
        ds: Path to file
        include_ties: Whether to include samples with tie (C) answers in calculation
        
    Returns:
        Dictionary containing metrics per dataset and overall average
    """


    # Calculate accuracies per dataset
    dataset_accuracies = {}

    for t, gt, dataset in zip(ds["text"], ds["gt"], ds["dataset"]):

        try:
            score = list(map(int, re.findall(r"<answer>(\d+)</answer>", t)))
            
            if score[0] > score[1]: pred = "A"
            elif score[0] < score[1]: pred = "B"
            else: pred = "C"
        except:
            pred = "C"

        # Skip ties if include_ties is False
        # if not include_ties and (gt == 'C' or pred == 'C'):
        if not include_ties and (gt == 'C'):
            continue

        if dataset not in dataset_accuracies:
            dataset_accuracies[dataset] = {
                'correct': 0,
                'total': 0
            }

        # Compare predictions with ground truth
        if pred == gt:
            dataset_accuracies[dataset]['correct'] += 1
        dataset_accuracies[dataset]['total'] += 1

    # Calculate metrics
    metrics = {}
    total_correct = 0
    total_samples = 0
    
    for dataset in dataset_accuracies:
        correct = dataset_accuracies[dataset]['correct']
        total = dataset_accuracies[dataset]['total']
        
        accuracy = correct / total if total > 0 else float('nan')
            
        metrics[dataset] = {
            'accuracy': accuracy,
            'num_samples': total,
            'correct': correct
        }
        
        total_correct += correct
        total_samples += total

    # Calculate overall accuracy
    metrics['overall'] = {
        'accuracy': total_correct / total_samples if total_samples > 0 else float('nan'),
        'num_samples': total_samples,
        'correct': total_correct
    }
    
    return metrics
    

In [ ]:
from collections import defaultdict


all_metrics_with_ties = defaultdict(list)
all_metrics_no_ties = defaultdict(list)


for ds in dss: # iterate over different runs (different seed)



    # Calculate metrics including ties
    metrics_with_ties = calculate_accuracy_metrics(ds, include_ties=True)

    print("\nResults including ties:")
    print("\nResults per dataset:")
    for dataset in metrics_with_ties:
        if dataset != 'overall':
            print(f"\n{dataset}:")
            print(f"Accuracy: {metrics_with_ties[dataset]['accuracy']:.3f}")
            print(f"Number of samples: {metrics_with_ties[dataset]['num_samples']}")
            print(f"Correct predictions: {metrics_with_ties[dataset]['correct']}")
            all_metrics_with_ties[dataset].append(metrics_with_ties[dataset]['accuracy'])
            
    print(f"\nOverall accuracy: {metrics_with_ties['overall']['accuracy']:.3f}")
    print(f"Total samples: {metrics_with_ties['overall']['num_samples']}")
    print(f"Total correct predictions: {metrics_with_ties['overall']['correct']}")
    all_metrics_with_ties['overall'].append(metrics_with_ties['overall']['accuracy'])

    # Calculate metrics excluding ties
    metrics_no_ties = calculate_accuracy_metrics(ds, include_ties=False)

    print("\nResults excluding ties:")
    print("\nResults per dataset:")
    for dataset in metrics_no_ties:
        if dataset != 'overall':
            print(f"\n{dataset}:")
            print(f"Accuracy: {metrics_no_ties[dataset]['accuracy']:.3f}")
            print(f"Number of samples: {metrics_no_ties[dataset]['num_samples']}")
            print(f"Correct predictions: {metrics_no_ties[dataset]['correct']}")
            all_metrics_no_ties[dataset].append(metrics_no_ties[dataset]['accuracy'])
            
    print(f"\nOverall accuracy: {metrics_no_ties['overall']['accuracy']:.3f}")
    print(f"Total samples: {metrics_no_ties['overall']['num_samples']}")
    print(f"Total correct predictions: {metrics_no_ties['overall']['correct']}")
    all_metrics_no_ties['overall'].append(metrics_no_ties['overall']['accuracy'])


    print(" ============================= \n")



dataset_list = ['coco', 'Concept Caption', 'diffusiondb', 'infographicsVQA', 'mathvista', 'textVQA', 'WIT', 'ChartQA', 'VisitBench', 'llava_bench', 'mind2web', 'ScienceQA',  'AesBench', 'mm-vet', 'overall']
assert set(all_metrics_with_ties.keys()) == set(dataset_list)
assert set(all_metrics_no_ties.keys()) == set(dataset_list)

print("\n\n +++++++++++ Final Results (Averaging) +++++++++++++ \n ")

print("With Tie:")
row_print = []
for i in dataset_list:
    v = np.mean(all_metrics_with_ties[i])
    print(i, ":", round(v, 3))
    row_print.append(round(v, 3).item())
print(row_print)

print("\nWithout Tie:")
row_print = []
for i in dataset_list:
    v = np.mean(all_metrics_no_ties[i])
    print(i, ":", round(v, 3))
    row_print.append(round(v, 3).item())
print(row_print)

## Batch

In [ ]:
dss = []

try:
    ds1 = load_dataset('json', data_files=f'{result_dir}/mllm-judge-{temperature}-{top_p}/{model}_{seed[0]}_batch.jsonl', split='train')
    dss.append(ds1)
    print('Loaded ds1')
except:
    print('Can not load ds1')
try:
    ds2 = load_dataset('json', data_files=f'{result_dir}/mllm-judge-{temperature}-{top_p}/{model}_{seed[1]}_batch.jsonl', split='train')
    dss.append(ds2)
    print('Loaded ds2')
except:
    print('Can not load ds2')

try:
    ds3 = load_dataset('json', data_files=f'{result_dir}/mllm-judge-{temperature}-{top_p}/{model}_{seed[2]}_batch.jsonl', split='train')
    dss.append(ds3)
    print('Loaded ds3')
except:
    print('Can not load ds3')


In [28]:
import json
import numpy as np
from Levenshtein import distance
import string

def calculate_levenshtein_metrics(ds):
    """
    Calculate edit distance metrics between predictions and ground truth
    
    Args:
        ds: Path to file
        
    Returns:
        Dictionary containing metrics per dataset and overall average
    """

    # Calculate distances per dataset
    dataset_distances = {}

    for t, gt, dataset in zip(ds['text'], ds['gt'], ds['dataset']):

        if dataset not in dataset_distances:
            dataset_distances[dataset] = []

        if gt is None or len(gt) <= 2:
            continue

        answers = re.findall(r"<answer>(.*?)</answer>", t, flags=re.DOTALL)
        v = [int(x) for block in answers for x in re.findall(r"\d+", block)][:len(gt)]
        labels = list(string.ascii_uppercase[:len(v)])
        sorted_labels = [label for _, label in sorted(zip(v, labels), reverse=True)]
        result = ''.join(sorted_labels)
        dist = distance(result, gt) / max(len(result), len(gt))
        
        
        dataset_distances[dataset].append(dist)
        
    # Calculate averages
    metrics = {}
    all_distances = []
    
    for dataset in dataset_distances:
        avg = np.mean(dataset_distances[dataset])
        metrics[dataset] = {
            'average_distance': avg,
            'num_samples': len(dataset_distances[dataset])
        }
        all_distances.extend(dataset_distances[dataset])
        
    metrics['overall'] = {
        'average_distance': np.mean(all_distances),
        'num_samples': len(all_distances)
    }
    
    return metrics 

In [ ]:
from collections import defaultdict

all_metrics = defaultdict(list)

for ds in dss:

    metrics = calculate_levenshtein_metrics(ds)

    # Print results
    print("\nResults per dataset:")
    for dataset in metrics:
        if dataset != 'overall':
            print(f"\n{dataset}:")
            print(f"Average Levenshtein distance: {metrics[dataset]['average_distance']:.2f}")
            print(f"Number of samples: {metrics[dataset]['num_samples']}")
            all_metrics[dataset].append(metrics[dataset]['average_distance'])
            
    print(f"\nOverall average distance: {metrics['overall']['average_distance']:.2f}")
    print(f"Total samples: {metrics['overall']['num_samples']}")
    all_metrics['overall'].append(metrics['overall']['average_distance'])

    print(" ============================= \n")




dataset_list = ['coco', 'Concept Caption', 'diffusiondb', 'infographicsVQA', 'mathvista', 'textVQA', 'WIT', 'ChartQA', 'VisitBench', 'llava_bench', 'mind2web', 'ScienceQA',  'AesBench', 'mm-vet', 'overall']
assert set(all_metrics.keys()) == set(dataset_list)

print("\n\n +++++++++++ Final Results (Averaging) +++++++++++++ \n ")

row_print = []
for i in dataset_list:
    v = np.mean(all_metrics[i])
    print(i, ":", round(v, 3))
    row_print.append(round(v, 3).item())
print(row_print)


## Score

In [ ]:
dss = []

try:
    ds1 = load_dataset('json', data_files=f'{result_dir}/mllm-judge-{temperature}-{top_p}/{model}_{seed[0]}_score.jsonl', split='train')
    dss.append(ds1)
    print('Loaded ds1')
except:
    print('Can not load ds1')
try:
    ds2 = load_dataset('json', data_files=f'{result_dir}/mllm-judge-{temperature}-{top_p}/{model}_{seed[1]}_score.jsonl', split='train')
    dss.append(ds2)
    print('Loaded ds2')
except:
    print('Can not load ds2')

try:
    ds3 = load_dataset('json', data_files=f'{result_dir}/mllm-judge-{temperature}-{top_p}/{model}_{seed[2]}_score.jsonl', split='train')
    dss.append(ds3)
    print('Loaded ds3')
except:
    print('Can not load ds3')

In [33]:
from scipy.stats import pearsonr
import math

def calculate_pearson_metrics(ds):
    """
    Calculate Pearson correlation metrics between predictions and ground truth scores
    
    Args:
        ds: Path to file
        
    Returns:
        Dictionary containing metrics per dataset and overall average
    """

    # Calculate correlations per dataset
    dataset_scores = {}

    for t, gt, dataset in zip(ds['text'], ds['gt'], ds['dataset']):

        if dataset not in dataset_scores:
            dataset_scores[dataset] = {
                'pred': [],
                'gt': []
            }

        v = list(map(int, re.findall(r"<answer>(\d+)</answer>", t)))
        if len(v) == 0: pred_score = -1
        else: 
            score_avg = sum(v) / len(v)
            score_avg = v[0]
            pred_score = math.ceil(score_avg/2)

        if pred_score > 5:
            pred_score = -1

        dataset_scores[dataset]['pred'].append(pred_score)
        dataset_scores[dataset]['gt'].append(int(gt))
        
    
    # Calculate metrics
    metrics = {}
    all_pred = []
    all_gt = []
    
    for dataset in dataset_scores:
        corr, p_value = pearsonr(dataset_scores[dataset]['pred'], dataset_scores[dataset]['gt'])
        metrics[dataset] = {
            'correlation': corr,
            'p_value': p_value,
            'num_samples': len(dataset_scores[dataset]['pred'])
        }
        all_pred.extend(dataset_scores[dataset]['pred'])
        all_gt.extend(dataset_scores[dataset]['gt'])
        
    # Calculate overall correlation
    overall_corr, overall_p = pearsonr(all_pred, all_gt)
    metrics['overall'] = {
        'correlation': overall_corr,
        'p_value': overall_p,
        'num_samples': len(all_pred)
    }
    
    return metrics


In [ ]:


from collections import defaultdict

all_metrics = defaultdict(list)

for ds in dss:

    pearson_metrics = calculate_pearson_metrics(ds)

    print("\nPearson Correlation Results:")
    print("\nResults per dataset:")
    for dataset in pearson_metrics:
        if dataset != 'overall':
            print(f"\n{dataset}:")
            print(f"Correlation: {pearson_metrics[dataset]['correlation']:.3f}")
            print(f"P-value: {pearson_metrics[dataset]['p_value']:.3e}")
            print(f"Number of samples: {pearson_metrics[dataset]['num_samples']}")
            all_metrics[dataset].append(pearson_metrics[dataset]['correlation'])
            
    print(f"\nOverall correlation: {pearson_metrics['overall']['correlation']:.3f}")
    print(f"Overall p-value: {pearson_metrics['overall']['p_value']:.3e}")
    print(f"Total samples: {pearson_metrics['overall']['num_samples']}")
    all_metrics['overall'].append(pearson_metrics['overall']['correlation'])

    print(" ============================= \n")



dataset_list = ['coco', 'Concept Caption', 'diffusiondb', 'infographicsVQA', 'mathvista', 'textVQA', 'WIT', 'ChartQA', 'VisitBench', 'llava_bench', 'mind2web', 'ScienceQA',  'AesBench', 'mm-vet', 'overall']
assert set(all_metrics.keys()) == set(dataset_list)

print("\n\n +++++++++++ Final Results (Averaging) +++++++++++++ \n ")

row_print = []
for i in dataset_list:
    v = np.mean(all_metrics[i])
    print(i, ":", round(v, 3))
    row_print.append(round(v, 3).item())
print(row_print)
